In [ ]:
!pip install qiskit

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import StatevectorSampler
import numpy as np

In [ ]:
shots=1024
qubits=3

In [ ]:
def measure(qc: QuantumCircuit):
    for i in range(qubits):
        qc.measure(i, i)

    sampler = StatevectorSampler()
    pm = generate_preset_pass_manager(optimization_level=1)

    isa = pm.run(qc)
    job = sampler.run([isa], shots=shots)
    pub = job.result()[0]
    return pub.data.c.get_counts()

def inversion_about_mean(qc: QuantumCircuit):
    # Inversion about mean circuit, derivation can be found in exercise sheet:
    # H^{⊗3} X^{⊗3} (X1 Z1 X1 Z1) CCZ_{1,2,3} X^{⊗3} H^{⊗3}

    qubits_list = list(range(qubits))

    qc.h(qubits_list)
    qc.x(qubits_list)

    qc.ccz(0, 1, 2)
    # - sign change
    for _ in range(2):
        qc.z(0)
        qc.x(0)

    qc.barrier()
    qc.x(qubits_list)
    qc.h(qubits_list)

def oracle(qc: QuantumCircuit):
    # 'hidden' oracle with |111⟩ as a marked state.
    # Derivation can be found in the exercise sheet.
    qc.barrier()
    qc.ccz(0,1,2)
    qc.barrier()

def construct_grover(qubits:int, k: int) -> QuantumCircuit:
    qc = QuantumCircuit(qubits, qubits)

    for i in range(qubits):
        qc.h(i)

    for i in range(k):
        oracle(qc)
        inversion_about_mean(qc)
    return qc

In [ ]:
# We test with applying Grover operator G k times and measure success probability
for k in range(1, 11):
    qc = construct_grover(qubits, k)
    counts = measure(qc)

    success_counts = counts['111']
    probability = success_counts / shots

    theta = np.arcsin(np.sqrt(1/2**qubits))
    prediction = np.sin((2*k+1)*theta)**2
    print(f"k={k:2d}  success state probability: {probability:.5f}   theoretical prediction: {prediction:.5f}")

k= 1  success state probability: 0.78418   theoretical prediction: 0.78125
k= 2  success state probability: 0.93945   theoretical prediction: 0.94531
k= 3  success state probability: 0.33984   theoretical prediction: 0.33008
k= 4  success state probability: 0.01074   theoretical prediction: 0.01221
k= 5  success state probability: 0.52051   theoretical prediction: 0.54797
k= 6  success state probability: 1.00000   theoretical prediction: 0.99979
k= 7  success state probability: 0.56543   theoretical prediction: 0.57697
k= 8  success state probability: 0.01855   theoretical prediction: 0.01946
k= 9  success state probability: 0.31543   theoretical prediction: 0.30289
k=10  success state probability: 0.91602   theoretical prediction: 0.93127
